In [2]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import plotly.graph_objects as go

# shared style
PALETTE = ["#2166ac", "#4393c3", "#92c5de", "#f4a582", "#d6604d", "#b2182b"]
plt.rcParams.update({
    "font.family": "sans-serif",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.dpi": 150,
})

DATA = "../../assets/data"

scatter = pd.read_json(f"{DATA}/scatter_distance_fare.json")
r2     = json.load(open(f"{DATA}/r2_comparison.json"))
bubble = pd.read_json(f"{DATA}/hhi_bubble.json")
mono   = pd.read_json(f"{DATA}/monopoly_routes.json")

Linked View Plot

In [3]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from ipywidgets import widgets, VBox, Output
from IPython.display import display, clear_output



airport_data = pd.read_json(f"{DATA}/airport_carriers.json")

us_airports = airport_data[
    (airport_data["LON"] > -130) &
    (airport_data["LON"] < -60) &
    (airport_data["LAT"] > 24) &
    (airport_data["LAT"] < 50)
].copy()

us_airports["CARRIER_COUNT"] = us_airports["carriers"].apply(len)


us_airports["PASSENGERS_LABEL"] = us_airports["TOTAL_PAX"].apply(lambda x: f"{x:,.0f}")



THEME = {
    "blue": "#2166ac",
    "blue_light": "#92c5de",
    "red": "#b2182b",
    "gray_dark": "#333333",
    "gray": "#666666",
    "gray_light": "#e6e6e6",
    "background": "#ffffff",
    "plot_background": "#f8f9fb",
}

# Widget

selected_airport = widgets.Dropdown(
    options=sorted(us_airports["AIRPORT"].unique()),
    value="ATL",
    description="Airport:",
    style={"description_width": "70px"},
    layout=widgets.Layout(width="300px")
)

#out = Output()
out = Output(layout=widgets.Layout(width="100%"))

# Linked plot function

# def make_linked_airport_plot(airport):
#     clear_output(wait=True)

def make_linked_airport_plot(airport):
    out.clear_output(wait=True)

    df = us_airports.copy()
    df["SELECTED"] = df["AIRPORT"] == airport

    selected_row = df[df["AIRPORT"] == airport].iloc[0]


## MAP
    map_fig = go.Figure()

    map_fig.add_trace(
        go.Scattergeo(
            lon=df["LON"],
            lat=df["LAT"],
            text=df["AIRPORT"],
            customdata=np.stack(
                [
                    df["CARRIER_COUNT"],
                    df["PASSENGERS_LABEL"]
                ],
                axis=-1
            ),
            mode="markers",
            marker=dict(
                size=np.sqrt(df["TOTAL_PAX"] / 175000),
                color=df["CARRIER_COUNT"],
                colorscale=[
                    [0.00, THEME["blue_light"]],
                    [1.00, THEME["blue"]]
                ],
                # colorbar=dict(
                #     title="Carrier Number",
                #     thickness=12,
                #     len=0.75,
                #     outlinewidth=0
                # ),
                colorbar=dict(
                    title="Carrier Number",
                    thickness=10,
                    len=0.6,
                    x=0.9   # pull it closer to plot
                ),
                line=dict(width=0.7, color="white"),
                opacity=0.72
            ),
            hovertemplate=(
                "<b>%{text}</b><br>"
                "Carrier Number: %{customdata[0]}<br>"
                "Passengers: %{customdata[1]}"
                "<extra></extra>"
            ),
            name="Airports"
        )
    )

    # Selected airport highlight
    map_fig.add_trace(
        go.Scattergeo(
            lon=[selected_row["LON"]],
            lat=[selected_row["LAT"]],
            text=[selected_row["AIRPORT"]],
            mode="markers+text",
            textposition="top center",
            marker=dict(
                size=22,
                color=THEME["red"],
                line=dict(width=2.5, color="white")
            ),
            textfont=dict(size=12, color=THEME["gray_dark"]),
            hovertemplate=(
                "<b>%{text}</b><br>"
                "Selected airport<br>"
                f"Carrier Number: {selected_row['CARRIER_COUNT']}<br>"
                f"Passengers: {selected_row['TOTAL_PAX']:,.0f}"
                "<extra></extra>"
            ),
            name="Selected Airport"
        )
    )

    map_fig.update_layout(
        title=dict(
            text=f"Airport Market Structure Map<br><sup>Selected airport: {airport}</sup>",
            x=0.02,
            xanchor="left"
        ),
        geo=dict(
            scope="usa",
            projection_type="albers usa",
            showland=True,
            landcolor=THEME["plot_background"],
            showcountries=False,
            showlakes=False,
            subunitcolor=THEME["gray_light"],
            bgcolor=THEME["background"]
        ),
        paper_bgcolor=THEME["background"],
        #width=850,
        autosize=True,
        height=600,
        margin=dict(l=20, r=20, t=70, b=20),
        legend=dict(
            orientation="h",
            y=-0.05,
            x=0,
            font=dict(size=11)
        )
    )

## SCATTERPLOT
    scatter_fig = go.Figure()

    scatter_fig.add_trace(
        go.Scatter(
            x=df["CARRIER_COUNT"],
            y=df["TOTAL_PAX"],
            text=df["AIRPORT"],
            customdata=np.stack(
                [
                    df["PASSENGERS_LABEL"],
                    df["SELECTED"]
                ],
                axis=-1
            ),
            mode="markers",
            marker=dict(
                size=np.where(df["SELECTED"], 18, 10),
                color=np.where(df["SELECTED"], THEME["red"], THEME["blue"]),
                opacity=np.where(df["SELECTED"], 1.0, 0.45),
                line=dict(
                    width=np.where(df["SELECTED"], 2.5, 0.6),
                    color="white"
                )
            ),
            hovertemplate=(
                "<b>%{text}</b><br>"
                "Carrier Number: %{x}<br>"
                "Passengers: %{customdata[0]}"
                "<extra></extra>"
            ),
            name="Airports"
        )
    )

    scatter_fig.add_annotation(
        x=selected_row["CARRIER_COUNT"],
        y=selected_row["TOTAL_PAX"],
        text=airport,
        showarrow=True,
        arrowhead=2,
        ax=35,
        ay=-35,
        bgcolor="white",
        bordercolor=THEME["gray_light"],
        borderwidth=1,
        font=dict(size=12, color=THEME["gray_dark"])
    )

    scatter_fig.update_layout(
        title=dict(
            text="Passenger Volume vs. Number of Carriers<br><sup>The selected airport is highlighted in red</sup>",
            x=0.02,
            xanchor="left"
        ),
        xaxis=dict(
            title="Number of Carriers",
            showgrid=True,
            gridcolor=THEME["gray_light"],
            zeroline=False
        ),
        yaxis=dict(
            title="Total Passengers",
            showgrid=True,
            gridcolor=THEME["gray_light"],
            zeroline=False,
            tickformat=","
        ),
        plot_bgcolor=THEME["plot_background"],
        paper_bgcolor=THEME["background"],
        #width=850,
        autosize=True,
        height=600,
        margin=dict(l=70, r=30, t=70, b=60),
        showlegend=False
    )

    # display(map_fig)
    # display(scatter_fig)

    map_fig.show(config={"responsive": True})
    scatter_fig.show(config={"responsive": True})

# Interaction

def update_plot(change):
    with out:
        make_linked_airport_plot(change["new"])

selected_airport.observe(update_plot, names="value")

#display(VBox([selected_airport, out]))
display(VBox(
    [selected_airport, out],
    layout=widgets.Layout(width="100%")
))

with out:
    make_linked_airport_plot(selected_airport.value)